## Import Libraries:

In [2]:
# Import Libraries
import pandas as pd
import re
import nltk
import string
import pickle

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Load Dataset:


In [3]:
# Load Dataset
df = pd.read_csv("IMDB Dataset.csv")  
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Data Preprocessing:

In [4]:
# Text Cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<br />", " ", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = text.split()
    stop_words = set(stopwords.words("english"))
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df["clean_review"] = df["review"].apply(clean_text)
df["label"] = df["sentiment"].apply(lambda x: 1 if x == "positive" else 0)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   review        50000 non-null  object
 1   sentiment     50000 non-null  object
 2   clean_review  50000 non-null  object
 3   label         50000 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 1.5+ MB


## Data Splitting:

In [5]:
# Train-Test Split
X = df["clean_review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## Vectorization:

In [6]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


## Model Training:

In [7]:
# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Naive Bayes": MultinomialNB(),
    "SVM": LinearSVC()
}

results = {}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results[name] = (acc, f1)
    print(f"{name} → Accuracy: {acc:.4f}, F1: {f1:.4f}")

Logistic Regression → Accuracy: 0.8914, F1: 0.8936
Naive Bayes → Accuracy: 0.8550, F1: 0.8566
SVM → Accuracy: 0.8821, F1: 0.8841


## Choosing Best model:

In [9]:

# Choose Best Model
best_model_name = max(results, key=lambda k: results[k][0])
print(f"\n Best Model: {best_model_name} with Accuracy {results[best_model_name][0]:.4f}")

best_model = models[best_model_name]


 Best Model: Logistic Regression with Accuracy 0.8914


## Model Saving:

In [10]:
# Save Model & Vectorizer
with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
